### Transform data to 28 x 28

In [123]:
from torchvision import datasets, transforms
import torch

# Data transformations
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

# Load the dataset
train_set = datasets.ImageFolder(root="../../dataset/train", transform=transform)
test_set = datasets.ImageFolder(root='../../dataset/test', transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=16, shuffle=False)

In [172]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Define the simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Input channels = 3 (RGB), output channels = 16
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=26*26*32, out_features=512)
        #self.drop = nn.Dropout(0.15)
        self.fc2 = nn.Linear(in_features=512, out_features=6)
        #self.fc3 = nn.Linear(in_features=128, out_features=6)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.flat(x)
        x = F.relu(self.fc1(x))
        #x = self.drop(x)
        x = self.fc2(x)
        #x = self.fc3(x)
        return x

model = SimpleCNN()
model

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=21632, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=6, bias=True)
)

In [173]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.04)

In [191]:
n_epochs = 10
for epoch in range(n_epochs):  # Train for 5 epochs
    running_loss = 0.0
    for images, labels in train_loader:
        # Move images and labels to the device

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {running_loss / len(train_loader):.4f}')
     


Epoch [1/10], Loss: 0.2583
Epoch [2/10], Loss: 0.1560
Epoch [3/10], Loss: 0.2759
Epoch [4/10], Loss: 0.0685
Epoch [5/10], Loss: 0.0411
Epoch [6/10], Loss: 0.3168
Epoch [7/10], Loss: 0.2525
Epoch [8/10], Loss: 0.0897
Epoch [9/10], Loss: 0.0257
Epoch [10/10], Loss: 0.0252


In [192]:
correct = 0
total = 0
with torch.no_grad():  # Disable gradient calculation for evaluation
    for images, labels in test_loader:
        # Move images and labels to the device

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')
     


Accuracy: 61.63%
